Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 04 · Factor exposures

**Question:** Does SMH capture dependence left after standard equity factors?

Security excess returns and six standard factors plus SMH share one complete observation mask per security/formation. Four intercept regressions use 504/400 and 252/200 window/count rules. The conditional SMH slope is the thematic exposure.

Research report · Method and code guide · Evidence map


**Chapter route**

Measurement and interpretation principles · Data and design context · Hero calculations and the result they produce · Read and run this notebook · Calculation source: factor exposures · Results, examples and interpretation · Worked example: recover a known conditional exposure · Visual evidence · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

OLS decomposes daily excess returns into fitted common-factor dependence and residual variation. The main conditional SMH loading comes from FF5 plus momentum plus SMH excess return, using the same per-security observation mask as the restricted regressions. Residualizing SMH against the baseline gives an equivalent conditional loading. Formation residual volatility is $\sqrt{252\, SSE/(n-\operatorname{rank}(X))}$, accounting for fitted regression degrees of freedom.


## Data and design context

All four daily excess-return regressions include an intercept and share the exact same observations for each security and formation. The standard model uses FF5 plus French momentum; the extended model also includes SMH excess return. Main estimates use 504 sessions with at least 400 complete observations, and sensitivity estimates use 252 sessions with at least 200. Shared security-history and quarantine rules determine eligibility.

The September 2026 formation is measured at August 31 for these factor estimates. Missing factor dates are never filled. Coefficients and residual variances are saved separately by window so later evaluation can apply them to future returns without refitting. Residual risk uses SSE divided by residual degrees of freedom. Standard errors are classical OLS conditional diagnostics, not the dependence-robust inference for the study's forward tests.

The residualization check estimates the SMH increment after removing standard-factor variation using the identical security-specific mask. Higher fitted R-squared is a diagnostic, not evidence of forecasting value. SMH's holdings overlap the stock universe, so some exposure is mechanical.

To rebuild all saved estimates, run `python -I -m quiet_index_500.factor_exposures` from the project virtual environment. This notebook reads saved summaries; the package contains all estimation logic.


## Hero calculations and the result they produce

`factor_exposures.estimate_factor_models` constructs the common mask, fits the four OLS designs, and checks the conditional SMH slope by residualization. `_ols` uses SSE/(n−rank) for residual variance and classical slope errors. `saved_factor_summary` below only reads results; it does not refit models.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version.


## Calculation source: factor exposures

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Quarterly security factor estimates on one common observation mask per security.

Standard errors are classical OLS conditional diagnostics, not dependence-robust
research inference. Every model includes an intercept. Residual risk uses SSE /
(n - rank), while residual variance share uses SSE / centered total sum of squares.
The four nested models share exactly the same stock, factor and SMH observations.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import calendar, eligibility, loader
from quiet_index_500.research_io import save_table

standard_factor_names = ("mkt_rf", "smb", "hml", "rmw", "cma", "mom")
model_factor_names = {
    "market": ("mkt_rf",),
    "market_smh": ("mkt_rf", "smh_excess"),
    "ff5_mom": standard_factor_names,
    "ff5_mom_smh": (*standard_factor_names, "smh_excess"),
}
model_prefixes = {"market": "m1", "market_smh": "m2", "ff5_mom": "m3", "ff5_mom_smh": "m4"}


**_ols**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [2]:
def _ols(response, design):
    coefficient, _, rank, _ = np.linalg.lstsq(design, response, rcond=None)
    if rank != design.shape[1]:
        raise ValueError("rank-deficient factor design")
    residual = response - design @ coefficient
    residual_sum_squares = float(residual @ residual)
    residual_degrees_freedom = len(response) - rank
    if residual_degrees_freedom <= 0:
        raise ValueError("insufficient residual degrees of freedom")
    residual_variance = residual_sum_squares / residual_degrees_freedom
    coefficient_covariance = residual_variance * np.linalg.inv(design.T @ design)
    standard_error = np.sqrt(np.maximum(np.diag(coefficient_covariance), 0))
    centered_response = response - response.mean()
    total_sum_squares = float(centered_response @ centered_response)
    r_squared = 1 - residual_sum_squares / total_sum_squares if total_sum_squares > 0 else np.nan
    adjusted_r_squared = 1 - (1 - r_squared) * (len(response) - 1) / residual_degrees_freedom
    return {"coefficient": coefficient, "standard_error": standard_error, "residual": residual,
            "nobs": len(response), "rank": rank, "residual_degrees_freedom": residual_degrees_freedom,
            "residual_variance": residual_variance, "residual_sample_variance": residual_sum_squares / (len(response) - 1),
            "r_squared": r_squared, "adjusted_r_squared": adjusted_r_squared,
            "residual_share_of_variance": 1 - r_squared}


**estimate_factor_models**

Fit all models for one security on identical dates; return fits and FWL diagnostics.

Inputs are a date-indexed decimal excess-return Series and a date-indexed frame
holding FF5, mom and smh_excess. The caller imposes the shared security-history
eligibility boundary before invoking this numerical estimator.


In [3]:
def estimate_factor_models(security_excess_returns, factors_with_smh, minimum_observations):
    """Fit all models for one security on identical dates; return fits and FWL diagnostics.

    Inputs are a date-indexed decimal excess-return Series and a date-indexed frame
    holding FF5, mom and smh_excess. The caller imposes the shared security-history
    eligibility boundary before invoking this numerical estimator.
    """
    required_names = list(model_factor_names["ff5_mom_smh"])
    joint = factors_with_smh[required_names].reindex(security_excess_returns.index).copy()
    joint.insert(0, "security", security_excess_returns)
    joint = joint.replace([np.inf, -np.inf], np.nan).dropna()
    if len(joint) < minimum_observations:
        raise ValueError("insufficient complete common observations")
    response = joint["security"].to_numpy(dtype=float)
    fits = {}
    for model, names in model_factor_names.items():
        design = np.column_stack([np.ones(len(joint)), joint[list(names)].to_numpy(dtype=float)])
        fits[model] = _ols(response, design)
        fits[model]["terms"] = ("const", *names)
    standard_design = np.column_stack([np.ones(len(joint)), joint[list(standard_factor_names)].to_numpy(dtype=float)])
    semiconductor_fit = _ols(joint["smh_excess"].to_numpy(dtype=float), standard_design)
    semiconductor_residual = semiconductor_fit["residual"]
    semiconductor_residual_sum_squares = float(semiconductor_residual @ semiconductor_residual)
    fwl_loading = float(semiconductor_residual @ fits["ff5_mom"]["residual"] / semiconductor_residual_sum_squares)
    joint_loading = float(fits["ff5_mom_smh"]["coefficient"][-1])
    return {"models": fits, "observation_dates": joint.index,
            "fwl_loading": fwl_loading, "fwl_difference": joint_loading - fwl_loading,
            "smh_factor_r_squared": semiconductor_fit["r_squared"],
            "smh_residual_volatility": np.sqrt(semiconductor_fit["residual_variance"] * settings.annualisation_sessions),
            "smh_residual_variance": semiconductor_fit["residual_variance"]}


**apply_formation_coefficients**

Apply one formation's saved long coefficients to later decimal factor returns.

Returns a date-by-ticker fitted excess-return frame. No fitting occurs. The
coefficient table must contain exactly one formation and at most one window;
callers select the intended vintage explicitly. Missing required factors yield
missing predictions. Subtract the result from future security excess returns
to obtain the fixed-coefficient residuals used by forward evaluation.


In [4]:
def apply_formation_coefficients(coefficients, factor_returns, model="ff5_mom"):
    """Apply one formation's saved long coefficients to later decimal factor returns.

    Returns a date-by-ticker fitted excess-return frame. No fitting occurs. The
    coefficient table must contain exactly one formation and at most one window;
    callers select the intended vintage explicitly. Missing required factors yield
    missing predictions. Subtract the result from future security excess returns
    to obtain the fixed-coefficient residuals used by forward evaluation.
    """
    if model not in model_factor_names:
        raise ValueError(f"unknown model {model}")
    selected = coefficients.loc[coefficients["model"] == model].copy()
    if selected.empty:
        return pd.DataFrame(index=factor_returns.index)
    for key in ("formation_date", "window_sessions"):
        if key in selected and selected[key].nunique(dropna=False) != 1:
            raise ValueError(f"select exactly one {key} before applying coefficients")
    terms = ["const", *model_factor_names[model]]
    coefficient_matrix = selected.pivot(index="term", columns="ticker", values="coefficient").reindex(terms)
    if coefficient_matrix.isna().any().any():
        raise ValueError("incomplete formation coefficient vector")
    factor_design = factor_returns[list(model_factor_names[model])].copy()
    factor_design.insert(0, "const", 1.0)
    values = factor_design.to_numpy(dtype=float) @ coefficient_matrix.to_numpy(dtype=float)
    return pd.DataFrame(values, index=factor_returns.index, columns=coefficient_matrix.columns)


**estimate_window_panel**

Estimate every security at all formations for one specified window.

Returns exposures (including ineligible rows), reusable long coefficients,
per-model diagnostics and a compact formation summary. All data enter through
the shared loader, with min(formation, factor cutoff) applied explicitly.


In [5]:
def estimate_window_panel(window_sessions, minimum_observations, *, apply_corrections=True):
    """Estimate every security at all formations for one specified window.

    Returns exposures (including ineligible rows), reusable long coefficients,
    per-model diagnostics and a compact formation summary. All data enter through
    the shared loader, with min(formation, factor cutoff) applied explicitly.
    """
    price_panels = loader.load_price_panels(apply_corrections=apply_corrections)
    returns = loader.daily_simple_returns_from_prices(price_panels.adjusted_close, price_panels.quarantined_returns)
    factors = loader.load_factors()
    excess_returns = loader.excess_returns(returns, factors["rf"])
    common_factors = factors[list(standard_factor_names)].join(loader.load_smh_excess_returns().rename("smh_excess"))
    identifiers = loader.load_identifiers()
    factor_cutoff = loader.factor_data_last_date()
    sessions = price_panels.adjusted_close.index
    exposure_rows, coefficient_rows, diagnostic_rows = [], [], []
    for formation_date in calendar.formation_dates(sessions):
        cutoff = calendar.measurement_cutoff(formation_date, factor_cutoff)
        window = calendar.estimation_window(sessions, cutoff, window_sessions)
        eligibility_table = eligibility.panel_window_eligibility(
            excess_returns, window, minimum_observations, common_factors,
            price_panels.before_same_security_start, price_panels.quarantined_returns)
        window_factors = common_factors.reindex(window)
        for ticker, eligibility_row in eligibility_table.iterrows():
            common = {"formation_date": formation_date, "ticker": ticker, "window_sessions": window_sessions}
            exposure = {**common, "cik": int(identifiers.at[ticker, "cik"]),
                        "is_representative_security": bool(identifiers.at[ticker, "is_representative_security"]),
                        "snapshot_weight": float(identifiers.at[ticker, "snapshot_weight"]),
                        "measurement_cutoff": cutoff, "window_first_session": window.min(), "window_last_session": window.max(),
                        "minimum_observations": minimum_observations, **eligibility_row.to_dict()}
            if not eligibility_row["eligible"]:
                exposure_rows.append(exposure)
                continue
            estimation = estimate_factor_models(excess_returns[ticker].reindex(window), window_factors, minimum_observations)
            if len(estimation["observation_dates"]) != eligibility_row["observations"]:
                raise AssertionError("regression and shared eligibility masks disagree")
            if abs(estimation["fwl_difference"]) > 1e-9:
                raise AssertionError("joint and same-mask residualized semiconductor estimates disagree")
            for model, fit in estimation["models"].items():
                prefix = model_prefixes[model]
                for term, coefficient, standard_error in zip(fit["terms"], fit["coefficient"], fit["standard_error"]):
                    exposure[f"{prefix}_{'alpha' if term == 'const' else term}"] = float(coefficient)
                    coefficient_rows.append({**common, "measurement_cutoff": cutoff, "model": model, "term": term,
                                             "coefficient": float(coefficient), "standard_error": float(standard_error), "nobs": fit["nobs"]})
                diagnostic_rows.append({**common, "measurement_cutoff": cutoff, "model": model,
                                        **{name: fit[name] for name in ("nobs", "rank", "residual_degrees_freedom", "r_squared", "adjusted_r_squared",
                                                                      "residual_variance", "residual_sample_variance", "residual_share_of_variance")}})
                exposure[f"{prefix}_r_squared"] = fit["r_squared"]
                exposure[f"{prefix}_adjusted_r_squared"] = fit["adjusted_r_squared"]
                exposure[f"{prefix}_residual_variance"] = fit["residual_variance"]
            baseline = estimation["models"]["ff5_mom"]
            exposure.update({"trailing_residual_volatility": np.sqrt(baseline["residual_variance"] * settings.annualisation_sessions),
                             "residual_share_of_variance": baseline["residual_share_of_variance"],
                             "smh_factor_r_squared": estimation["smh_factor_r_squared"],
                             "smh_residual_volatility": estimation["smh_residual_volatility"],
                             "smh_residual_variance": estimation["smh_residual_variance"],
                             "fwl_loading": estimation["fwl_loading"], "fwl_difference": estimation["fwl_difference"],
                             "incremental_r_squared": estimation["models"]["ff5_mom_smh"]["r_squared"] - baseline["r_squared"]})
            exposure_rows.append(exposure)
    exposures = pd.DataFrame(exposure_rows).sort_values(["ticker", "formation_date"])
    loading_columns = [f"{model_prefixes[model]}_{factor}" for model, names in model_factor_names.items() for factor in names]
    for column in loading_columns:
        exposures[f"{column}_change"] = exposures.groupby("ticker", sort=False)[column].diff()  # no bridge across an ineligible formation
    exposures = exposures.sort_values(["formation_date", "ticker"]).reset_index(drop=True)
    coefficients = pd.DataFrame(coefficient_rows)
    diagnostics = pd.DataFrame(diagnostic_rows)
    summaries = []
    for formation_date, rows in exposures.groupby("formation_date", sort=True):
        eligible_rows = rows.loc[rows["eligible"]]
        summaries.append({"formation_date": formation_date, "window_sessions": window_sessions,
                          "measurement_cutoff": rows["measurement_cutoff"].iloc[0], "securities": len(rows), "eligible": len(eligible_rows),
                          "eligible_representatives": int(eligible_rows["is_representative_security"].sum()),
                          "represented_snapshot_weight": eligible_rows["snapshot_weight"].sum(),
                          "median_smh_loading": eligible_rows["m4_smh_excess"].median(),
                          "median_incremental_r_squared": eligible_rows["incremental_r_squared"].median(),
                          "median_smh_factor_r_squared": eligible_rows["smh_factor_r_squared"].median(),
                          "median_smh_residual_volatility": eligible_rows["smh_residual_volatility"].median(),
                          "maximum_absolute_fwl_difference": eligible_rows["fwl_difference"].abs().max()})
    return {"exposures": exposures, "coefficients": coefficients, "diagnostics": diagnostics, "formation_summary": pd.DataFrame(summaries)}


**build_factor_exposures**

Save both specified windows; return a mapping of output names to saved paths.


In [6]:
def build_factor_exposures():
    """Save both specified windows; return a mapping of output names to saved paths."""
    paths = {}
    for window, minimum in ((settings.main_window_sessions, settings.main_window_minimum_observations),
                            (settings.sensitivity_window_sessions, settings.sensitivity_window_minimum_observations)):
        tables = estimate_window_panel(window, minimum)
        for name, table in tables.items():
            path = save_table(table, "factor_exposures", f"{name}_{window}.csv", source_modules=[settings.repository_root_folder / 'src/quiet_index_500/factor_exposures.py'],
                              note="Four intercept OLS models on identical per-security observations. Standard errors are classical conditional diagnostics; "
                                   "residual variance uses SSE/(n-rank). Factor estimates stop at min(formation date, observed factor cutoff).")
            paths[f"{name}_{window}"] = path
        print(f"factor exposures: saved {window}-session window, {int(tables['exposures']['eligible'].sum())} eligible security-formations", flush=True)
    return paths


**saved_factor_summary**

Read the saved compact summary for notebook display without rerunning estimation.


In [7]:
def saved_factor_summary():
    """Read the saved compact summary for notebook display without rerunning estimation."""
    return pd.concat([pd.read_csv(settings.results_folder / "factor_exposures" / f"formation_summary_{window}.csv")
                      for window in (settings.main_window_sessions, settings.sensitivity_window_sessions)], ignore_index=True)


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [8]:
summary = saved_factor_summary()
summary

,formation_date,window_sessions,measurement_cutoff,securities,eligible,eligible_representatives,represented_snapshot_weight,median_smh_loading,median_incremental_r_squared,median_smh_factor_r_squared,median_smh_residual_volatility,maximum_absolute_fwl_difference
0,2017-12-29,504,2017-12-29,503,461,459,0.954222,-0.027776,0.003664,0.641434,0.116484,8.326673e-16
1,2018-03-29,504,2018-03-29,503,463,461,0.956940,-0.031610,0.003683,0.658076,0.116332,1.110223e-15
2,2018-06-29,504,2018-06-29,503,464,462,0.957602,-0.037244,0.002585,0.661095,0.116401,8.881784e-16
3,2018-09-28,504,2018-09-28,503,466,464,0.959184,-0.031383,0.002164,0.628901,0.121596,9.436896e-16
4,2018-12-31,504,2018-12-31,503,468,466,0.960261,-0.036618,0.002749,0.696939,0.126766,2.220446e-15
...,...,...,...,...,...,...,...,...,...,...,...,...
67,2025-09-30,252,2025-09-30,503,498,495,0.994719,-0.152673,0.014522,0.818233,0.159472,2.220446e-15
68,2025-12-31,252,2025-12-31,503,499,496,0.998572,-0.158957,0.016469,0.824837,0.160527,1.998401e-15
69,2026-03-31,252,2026-03-31,503,499,496,0.998572,-0.121647,0.011256,0.804820,0.164704,1.110223e-15
70,2026-06-30,252,2026-06-30,503,499,496,0.998572,-0.132885,0.012717,0.765436,0.173360,1.110223e-15


In [9]:
summary.loc[summary['window_sessions'] == 504].tail(8)

,formation_date,window_sessions,measurement_cutoff,securities,eligible,eligible_representatives,represented_snapshot_weight,median_smh_loading,median_incremental_r_squared,median_smh_factor_r_squared,median_smh_residual_volatility,maximum_absolute_fwl_difference
28,2024-12-31,504,2024-12-31,503,493,490,0.989670,-0.066844,0.008518,0.664374,0.183271,9.992007e-16
29,2025-03-31,504,2025-03-31,503,493,490,0.989670,-0.074811,0.008526,0.683233,0.183857,1.193490e-15
30,2025-06-30,504,2025-06-30,503,494,491,0.990019,-0.089257,0.008549,0.762183,0.176497,9.992007e-16
31,2025-09-30,504,2025-09-30,503,494,491,0.990019,-0.085005,0.008567,0.754616,0.176849,1.110223e-15
32,2025-12-31,504,2025-12-31,503,497,494,0.994368,-0.111380,0.010052,0.764384,0.177678,2.442491e-15
33,2026-03-31,504,2026-03-31,503,497,494,0.994368,-0.116483,0.011187,0.775918,0.175426,8.881784e-16
34,2026-06-30,504,2026-06-30,503,497,494,0.994368,-0.139072,0.014048,0.783839,0.183855,1.054712e-15
35,2026-09-30,504,2026-08-31,503,498,495,0.994719,-0.139496,0.014451,0.771739,0.187012,2.720046e-15


In [10]:
summary.loc[summary['window_sessions'] == 252].tail(8)

,formation_date,window_sessions,measurement_cutoff,securities,eligible,eligible_representatives,represented_snapshot_weight,median_smh_loading,median_incremental_r_squared,median_smh_factor_r_squared,median_smh_residual_volatility,maximum_absolute_fwl_difference
64,2024-12-31,252,2024-12-31,503,494,491,0.990019,-0.081214,0.009839,0.727529,0.184594,1.554312e-15
65,2025-03-31,252,2025-03-31,503,497,494,0.994368,-0.091439,0.010601,0.768655,0.179162,1.332268e-15
66,2025-06-30,252,2025-06-30,503,497,494,0.994368,-0.118885,0.012642,0.855855,0.164556,1.776357e-15
67,2025-09-30,252,2025-09-30,503,498,495,0.994719,-0.152673,0.014522,0.818233,0.159472,2.220446e-15
68,2025-12-31,252,2025-12-31,503,499,496,0.998572,-0.158957,0.016469,0.824837,0.160527,1.998401e-15
69,2026-03-31,252,2026-03-31,503,499,496,0.998572,-0.121647,0.011256,0.804820,0.164704,1.110223e-15
70,2026-06-30,252,2026-06-30,503,499,496,0.998572,-0.132885,0.012717,0.765436,0.173360,1.110223e-15
71,2026-09-30,252,2026-08-31,503,501,498,0.999057,-0.121801,0.010050,0.821665,0.168078,2.664535e-15


## Worked example: recover a known conditional exposure

Illustrative inputs are separate from study data. Re-run this short cell to connect the method to its numerical output.


In [11]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(41)
example_factors = pd.DataFrame(rng.normal(0, .01, (504, 7)), columns=model_factor_names['ff5_mom_smh'])
example_returns = .0002 + .8 * example_factors.mkt_rf + .6 * example_factors.smh_excess
example_fit = estimate_factor_models(example_returns, example_factors, 400)
pd.Series({'joint_smh_loading': example_fit['models']['ff5_mom_smh']['coefficient'][-1],
           'residualized_loading': example_fit['fwl_loading']})

joint_smh_loading       0.6
residualized_loading    0.6
dtype: float64

## Visual evidence

Source figure: 02_exposure_stability

These figures use saved results; see the figure guide for sources, sample definitions and limitations.


## Findings

The mean formation-median incremental fitted R² is about 0.005. Joint and same-mask residualized estimates agree numerically. This establishes additional historical fit, not future forecasting value.

## Analysis and takeaways

The mean of formation-level median incremental R-squared is about 0.005. This measures historical explanatory fit, not future forecast accuracy or causation. Signed loading captures direction; common variance can arise for either sign. Adjacent overlapping windows also induce descriptive persistence.

## Bridge through the study

07 uses representative-security characteristics. 08 uses saved standard-factor coefficients to define future residual risk. 09 uses eligibility and SMH loading for portfolio membership, then refits both models on portfolio-common training dates.
